In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Sales Table 
- products_sold = main grain table which has sales record
- this can be used as reference table for all sales related analytics table

In [0]:
fact_sales_target = "zara_lakehouse.retail_silver.fact_sales_silver"

In [0]:
df = spark.sql("""
               
SELECT 
    ps.product_iD,
    ps.city,
    ps.date,
    ps.pieces_sold,

    -- bring price from product table
    pi.price AS unit_price,

    -- revenue
    (ps.pieces_sold * pi.price) AS revenue,

    -- bring discount (LEFT JOIN because not all cities may exist or date doesnt exit for our grain)
    COALESCE(d.discount, 0) AS discount,

    -- discount amount
    (ps.pieces_sold * pi.price * COALESCE(d.discount, 0)) AS discount_amount,

    -- net revenue
    (ps.pieces_sold * pi.price * (1 - COALESCE(d.discount, 0))) AS net_revenue

FROM zara_lakehouse.retail_silver.dim_products_sold_silver ps

JOIN zara_lakehouse.retail_silver.dim_product_information_silver pi 
    ON ps.product_id = pi.product_id

LEFT JOIN zara_lakehouse.retail_silver.dim_discount_silver d
    ON ps.product_id = d.product_id
    AND ps.city = d.city
    
    """)

In [0]:
clean_fact_sales_silver_df = df.write \
        .mode("overwrite") \
        .option ("mergeSchema", "true") \
        .saveAsTable(fact_sales_target)